In [29]:
import os,sys
#sys.path.append('/work/qdiff/mo_utils')

In [30]:
from mo_utils.utils.tmux_utils import get_session_list,tmux_session,get_session_name,kill_session
from pathlib import Path

In [31]:
get_session_list()

[Session($0 qdiff_ver_test_iandh_exp2_har_all_cuda_0),
 Session($1 qdiff_ver_test_iandh_exp4_har_all_cuda_1),
 Session($2 sd_quantize_qm=qdiff_gpu_0),
 Session($3 sd_quantize_qm=qdiff_gpu_1)]

In [4]:
#kill_session()

In [5]:
#w8bit_sym,nbit,symmetric = '/fastdata/users/nadavg/sd/qdiff/output_quantization/2025-01-26-18-44-03/ckpt.pth',8,True

In [32]:
nbit,symmetric = 8,True

In [36]:
task = 'quantize'
gpu = 5
prompt = "a puppy wearing a hat" 
weight_bit = 8
symmetric_weight = True#True
bs = 8
act_bit = 8

outdir= "/workspace/sd/qdiff_hf15/output_quantization"
quant_act_ops = True #False #True#True
split_to_16bits = False

resume_w = False
accum_batches = True#True
quantized_ckpt_path = ''
#fp_model_path ="''" #"/genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/quantize_params_10_03/unet_sim_eq.har"
fp_model_path ="/genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/quantize_params_10_03/unet_sim_eq.har"

#quantized_ckpt_path = w8bit_sym
quant_mode = 'qdiff'
naive_weights_quant =  "true"
rev_order =  "false"
unite_kvq_act = "true"
unite_skip_ln = "true"
split = "false"
sm_bit = 16 #16# 8 #8 #~!!!!!!!!!!!!!!!
act16bits_rtn = "true"
partial_sm_abit = "ver1"
channel_wise_weights = "false" #!!!



ddim_steps = 20
if ddim_steps == 20:
    cali_n= 128
    cali_st = 10
    cali_data_path = "/genai/users/nadavg/sd/qdiff_hf15_verd/gen_calib/calib_dict_steps20.pt"
    cali_iters = 20000#5000 
    cali_iters_a = 5000
    #act_bit = 16
elif ddim_steps == 50:
    cali_n= 128
    cali_st = 25
    cali_data_path = "/genai/users/nadavg/sd/qdiff_hf15_verd/gen_calib/calib_dict_steps50.pt"
    cali_iters = 20000#5000 
    cali_iters_a = 5000 
else:
    raise ValueError('ddim_steps must be 20 or 50')

#cali_data_path='/fastdata/users/nadavg/sd/qdiff/sd_coco-s75_sample1024_allst.pt'


debug = False#True#False


In [37]:
cmd=(f"python scripts/hf15/txt2img.py --prompt '{prompt}' --plms --cond --ptq --weight_bit {weight_bit} "+
    f"--quant_act --act_bit {act_bit} --cali_st {cali_st} --cali_batch_size {bs} --cali_n {cali_n} --no_grad_ckpt  --running_stat "+
    f"--sm_abit {sm_bit} --cali_data_path {cali_data_path} --outdir {outdir} --ddim_steps {ddim_steps}" +
    symmetric_weight*" --symmetric_weight "+
    resume_w*f"--resume_w --cali_ckpt {quantized_ckpt_path} "+
    quant_act_ops*" --quant_act_ops "+
    split_to_16bits*" --split_to_16bits "+
    accum_batches*" --accum_batches "+
    f"--quant_mode {quant_mode}"+
    f" --naive_weights_quant {naive_weights_quant} "+
    f" --rev_order {rev_order} "+
    f"--cali_iters {cali_iters} --cali_iters_a {cali_iters_a} "+
    f"--fp_model_path {fp_model_path} "+
    f"--unite_kvq_act {unite_kvq_act} "+
    f"--unite_skip_ln {unite_skip_ln} "+
    f"--act16bits_rtn {act16bits_rtn} "+
    f"--partial_sm_abit {partial_sm_abit} "+
    f"--channel_wise_weights {channel_wise_weights} "+
    f"--split {split} "+
    debug*" --debug "
    )


In [38]:
cmd

"python scripts/hf15/txt2img.py --prompt 'a puppy wearing a hat' --plms --cond --ptq --weight_bit 8 --quant_act --act_bit 8 --cali_st 10 --cali_batch_size 8 --cali_n 128 --no_grad_ckpt  --running_stat --sm_abit 16 --cali_data_path /genai/users/nadavg/sd/qdiff_hf15_verd/gen_calib/calib_dict_steps20.pt --outdir /workspace/sd/qdiff_hf15/output_quantization --ddim_steps 20 --symmetric_weight  --quant_act_ops  --accum_batches --quant_mode qdiff --naive_weights_quant true  --rev_order false --cali_iters 20000 --cali_iters_a 5000 --fp_model_path /genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/quantize_params_10_03/unet_sim_eq.har --unite_kvq_act true --unite_skip_ln true --act16bits_rtn true --partial_sm_abit ver1 --channel_wise_weights false --split false "

In [39]:
inst_list = [f'cd {Path.home() / "q-diffusion"}',
             "mobax",
             f'export CUDA_VISIBLE_DEVICES={gpu}',
             cmd]

In [40]:
inst_list

['cd /home/nadavg/q-diffusion',
 'mobax',
 'export CUDA_VISIBLE_DEVICES=5',
 "python scripts/hf15/txt2img.py --prompt 'a puppy wearing a hat' --plms --cond --ptq --weight_bit 8 --quant_act --act_bit 8 --cali_st 10 --cali_batch_size 8 --cali_n 128 --no_grad_ckpt  --running_stat --sm_abit 16 --cali_data_path /genai/users/nadavg/sd/qdiff_hf15_verd/gen_calib/calib_dict_steps20.pt --outdir /workspace/sd/qdiff_hf15/output_quantization --ddim_steps 20 --symmetric_weight  --quant_act_ops  --accum_batches --quant_mode qdiff --naive_weights_quant true  --rev_order false --cali_iters 20000 --cali_iters_a 5000 --fp_model_path /genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/quantize_params_10_03/unet_sim_eq.har --unite_kvq_act true --unite_skip_ln true --act16bits_rtn true --partial_sm_abit ver1 --channel_wise_weights false --split false "]

In [41]:
sess_name = get_session_name(f'sd_{task}_qm={quant_mode}_gpu_{gpu}')
sess_name = sess_name if not debug else sess_name + '_debug'

sess_name

'sd_quantize_qm=qdiff_gpu_5'

In [42]:
tmux_session(sess_name,inst_list)

tmux attach -t "sd_quantize_qm=qdiff_gpu_5"


'sd_quantize_qm=qdiff_gpu_5'

In [ ]:
#kill_session(kill_only=sess_name)

before kill sessions=[Session($23 qdiff_ver_test_iandh_exp2_har_all_cuda_1), Session($24 qdiff_ver_test_iandh_exp2_har_all_cuda_2), Session($25 sd_quantize_qm=qdiff_gpu_0)]
after kill sessions=[Session($23 qdiff_ver_test_iandh_exp2_har_all_cuda_1), Session($24 qdiff_ver_test_iandh_exp2_har_all_cuda_2)]
